# 02 — EDA del Dataset Procesado

Análisis del output de `python -m src.features.pipeline`.

**Requiere**: `data/processed/*.npy` (correr el pipeline primero).

Secciones:
1. Carga y estadísticas básicas
2. Distribución de labels y balance
3. Distribución de features numéricas (post-scaling)
4. Correlación entre features
5. Separabilidad: ¿discriminan las features entre Yes/No?
6. Análisis de categorías y embeddings de texto
7. Split temporal: distribución train/val

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.features.numerical import NUMERICAL_FEATURE_NAMES

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
PROCESSED = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'figures'

assert PROCESSED.exists(), f'No existe {PROCESSED} — correr: python -m src.features.pipeline'
print('Procesado OK')

## 1. Carga y estadísticas básicas

In [ ]:
X = np.load(PROCESSED / 'numerical_features.npy')    # (N, 23)
y = np.load(PROCESSED / 'labels.npy')                # (N,)
C = np.load(PROCESSED / 'category_ids.npy')          # (N,)
T = np.load(PROCESSED / 'text_embeddings.npy')        # (N, 384)
E = np.load(PROCESSED / 'end_dates.npy')             # (N,) timestamps

N, F = X.shape
print(f'Samples:              {N:,}')
print(f'Features numéricas:   {F}  (esperado: 23)')
print(f'Categorías únicas:    {len(np.unique(C))}')
print(f'Dimensión embedding:  {T.shape[1]}')
print(f'Positivos (label=1):  {y.sum():.0f} ({100*y.mean():.1f}%)')
print(f'Negativos (label=0):  {(1-y).sum():.0f} ({100*(1-y).mean():.1f}%)')
print(f'Ratio pos/neg:        {y.sum()/(1-y).sum():.2f}')

## 2. Distribución de labels y balance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bar chart
ax = axes[0]
counts = [int((y==0).sum()), int((y==1).sum())]
bars = ax.bar(['No (0)', 'Yes (1)'], counts, color=[PALETTE[3], PALETTE[2]], alpha=0.85, width=0.5)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{cnt:,}\n({100*cnt/N:.1f}%)', ha='center', fontsize=10)
ax.set_title('Balance de clases')
ax.set_ylabel('Samples')

# Distribución temporal de labels
ax = axes[1]
if E.max() > 0:
    dates = pd.to_datetime(E, unit='s', utc=True)
    df_temp = pd.DataFrame({'date': dates, 'label': y})
    df_temp['month'] = df_temp['date'].dt.to_period('M')
    monthly = df_temp.groupby('month')['label'].agg(['sum','count'])
    monthly['pct_yes'] = 100 * monthly['sum'] / monthly['count']
    ax.plot(monthly.index.astype(str), monthly['pct_yes'], color=PALETTE[2], marker='o', ms=4)
    ax.axhline(100*y.mean(), color='gray', linestyle='--', lw=1, label=f'Promedio {100*y.mean():.1f}%')
    ax.set_title('% Yes por mes (endDate)')
    ax.set_xlabel('Mes')
    ax.set_ylabel('% Yes')
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
else:
    ax.text(0.5, 0.5, 'Sin timestamps', ha='center', va='center', transform=ax.transAxes)

# Categorías
ax = axes[2]
cat_counts = pd.Series(C).value_counts().sort_index()
ax.bar(cat_counts.index, cat_counts.values, color=PALETTE[0], alpha=0.85)
ax.set_title('Distribución de categorías\n(IDs 0-9)')
ax.set_xlabel('Category ID')
ax.set_ylabel('Samples')

plt.suptitle('Dataset procesado — distribución de labels', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'processed_labels.png', bbox_inches='tight')
plt.show()

## 3. Distribución de features numéricas (post-scaling)

Después del StandardScaler, cada feature debería tener media ~0 y std ~1.
Features con % de ceros alto son normales (e.g., order book en resueltos).

In [ ]:
df_feat = pd.DataFrame(X, columns=NUMERICAL_FEATURE_NAMES)

# Estadísticas básicas
stats = df_feat.describe().T
stats['pct_zero'] = (df_feat == 0).mean() * 100
stats['pct_zero_str'] = stats['pct_zero'].round(1).astype(str) + '%'
print('=== Stats post-scaling ===')
print(stats[['mean','std','min','max','pct_zero_str']].round(3).to_string())

In [ ]:
# % de ceros por feature
zero_pct = (df_feat == 0).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
colors = ['#C44E52' if v > 60 else '#DD8452' if v > 20 else '#55A868' for v in zero_pct]
zero_pct.sort_values().plot.barh(ax=ax, color=colors, alpha=0.85)
ax.axvline(50, color='gray', linestyle='--', lw=1)
ax.set_title('% de ceros por feature\n(rojo>60%, naranja>20%, verde OK)', fontsize=10)
ax.set_xlabel('% de ceros')

# Distribuciones (violin) de features con pocos ceros
ax = axes[1]
low_zero = zero_pct[zero_pct < 30].index.tolist()[:8]
if low_zero:
    data_plot = [df_feat[c].values for c in low_zero]
    vp = ax.violinplot(data_plot, positions=range(len(low_zero)), showmedians=True)
    for body in vp['bodies']:
        body.set_facecolor(PALETTE[0])
        body.set_alpha(0.6)
    ax.set_xticks(range(len(low_zero)))
    ax.set_xticklabels(low_zero, rotation=45, ha='right', fontsize=8)
    ax.set_title('Distribución de features con <30% ceros\n(post-StandardScaler)')
    ax.axhline(0, color='gray', linestyle='--', lw=1)

plt.suptitle('Cobertura y distribución de features numéricas procesadas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'processed_feature_distributions.png', bbox_inches='tight')
plt.show()

## 4. Correlación entre features

In [ ]:
corr = df_feat.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax,
    annot_kws={'size': 6}, linewidths=0.3, square=True,
)
ax.set_title('Correlación entre features numéricas (post-scaling)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'processed_correlation.png', bbox_inches='tight')
plt.show()

# Pares más correlacionados
corr_upper = corr.where(~mask & (corr.abs() > 0.5))
print('Correlaciones altas (|r|>0.5):')
pairs = corr_upper.stack().sort_values(key=abs, ascending=False)
for (a, b), v in pairs.head(10).items():
    print(f'  {a:30s} x {b:30s}: {v:+.2f}')

## 5. Separabilidad: ¿discriminan las features entre Yes y No?

Para cada feature, calculamos el AUC-ROC univariado. Una feature perfecta tendría AUC=1.0; una inútil AUC=0.5.

In [ ]:
from sklearn.metrics import roc_auc_score

aucs = []
for i, name in enumerate(NUMERICAL_FEATURE_NAMES):
    col = X[:, i]
    if col.std() == 0 or len(np.unique(col)) < 2:
        aucs.append(0.5)
        continue
    try:
        auc = roc_auc_score(y, col)
        auc = max(auc, 1 - auc)  # tomar el mayor entre AUC y 1-AUC
    except Exception:
        auc = 0.5
    aucs.append(auc)

auc_series = pd.Series(aucs, index=NUMERICAL_FEATURE_NAMES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#C44E52' if v > 0.6 else '#55A868' if v > 0.55 else '#4C72B0' for v in auc_series]
auc_series.plot.barh(ax=ax, color=colors[::-1], alpha=0.85)
ax.axvline(0.5, color='gray', linestyle='--', lw=1.5, label='Random (0.5)')
ax.axvline(0.55, color='#DD8452', linestyle='--', lw=1, label='Útil (0.55)')
ax.axvline(0.6, color='#C44E52', linestyle='--', lw=1, label='Bueno (0.60)')
ax.set_xlabel('AUC-ROC univariado')
ax.set_title('Separabilidad por feature\n(AUC-ROC — cuánto discrimina Yes vs No sola)', fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(0.45, 0.85)
plt.tight_layout()
plt.savefig(FIGURES / 'processed_feature_auc.png', bbox_inches='tight')
plt.show()

print('Top features (AUC):')
for name, auc in auc_series.head(10).items():
    bar = '█' * int((auc - 0.5) * 100)
    print(f'  {name:30s}: {auc:.3f} {bar}')

## 6. Box plots bivariados: features vs label

In [ ]:
# Top 12 features por AUC para box plots
top_features = auc_series.head(12).index.tolist()

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for ax, feat in zip(axes, top_features):
    i = NUMERICAL_FEATURE_NAMES.index(feat)
    yes_v = X[y == 1, i]
    no_v  = X[y == 0, i]
    # Excluir ceros para features TS (más informativo)
    if 'momentum' in feat or 'volatility' in feat or 'trend' in feat or 'ewm' in feat:
        yes_v = yes_v[yes_v != 0]
        no_v  = no_v[no_v != 0]
    if len(yes_v) < 5 or len(no_v) < 5:
        ax.set_visible(False)
        continue
    bp = ax.boxplot([yes_v, no_v], patch_artist=True,
                   medianprops={'color': 'black', 'lw': 2},
                   flierprops={'marker': '.', 'markersize': 1.5, 'alpha': 0.3},
                   widths=0.4)
    bp['boxes'][0].set_facecolor(PALETTE[2] + 'AA')
    bp['boxes'][1].set_facecolor(PALETTE[3] + 'AA')
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Yes', 'No'])
    ax.set_title(f'{feat}\nAUC={auc_series[feat]:.3f}', fontsize=8)

for ax in axes[len(top_features):]:
    ax.set_visible(False)

plt.suptitle('Top 12 features por AUC-ROC — distribución Yes vs No', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'processed_bivariate_top.png', bbox_inches='tight')
plt.show()

## 7. PCA — visualización del espacio de features

In [ ]:
# PCA con las features que no son todas cero
non_zero_cols = [i for i, name in enumerate(NUMERICAL_FEATURE_NAMES)
                 if (X[:, i] != 0).mean() > 0.05]
X_nz = X[:, non_zero_cols]
names_nz = [NUMERICAL_FEATURE_NAMES[i] for i in non_zero_cols]
print(f'Features con >5% no-cero: {len(non_zero_cols)}')

pca = PCA(n_components=min(10, len(non_zero_cols)))
X_pca = pca.fit_transform(X_nz)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter PC1 vs PC2
ax = axes[0]
for label, color, name in [(0, PALETTE[3], 'No'), (1, PALETTE[2], 'Yes')]:
    mask_l = y == label
    ax.scatter(X_pca[mask_l, 0], X_pca[mask_l, 1],
               c=color, alpha=0.3, s=8, label=name)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — PC1 vs PC2')
ax.legend()

# Varianza explicada acumulada
ax = axes[1]
cum_var = np.cumsum(pca.explained_variance_ratio_) * 100
ax.plot(range(1, len(cum_var)+1), cum_var, marker='o', color=PALETTE[0])
ax.axhline(80, color='gray', linestyle='--', lw=1, label='80%')
ax.axhline(90, color='gray', linestyle=':', lw=1, label='90%')
ax.set_xlabel('Número de componentes')
ax.set_ylabel('Varianza explicada acumulada (%)')
ax.set_title('PCA — Varianza explicada')
ax.legend()

plt.suptitle('PCA del espacio de features numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / 'processed_pca.png', bbox_inches='tight')
plt.show()

print('Varianza por componente:')
for i, v in enumerate(pca.explained_variance_ratio_[:5], 1):
    print(f'  PC{i}: {v*100:.1f}%')

## 8. Split temporal — distribución de train vs val

In [ ]:
if E.max() > 0:
    n = len(y)
    n_val = int(n * 0.2)
    sorted_idx = np.argsort(E)
    train_idx = sorted_idx[:n - n_val]
    val_idx   = sorted_idx[n - n_val:]

    y_train = y[train_idx]
    y_val   = y[val_idx]
    dates_all = pd.to_datetime(E, unit='s', utc=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    ax.hist(dates_all[train_idx], bins=30, alpha=0.6, color=PALETTE[0], label='Train')
    ax.hist(dates_all[val_idx],   bins=30, alpha=0.6, color=PALETTE[1], label='Val')
    ax.set_title('Distribución temporal\n(train=más antiguos, val=más recientes)')
    ax.set_xlabel('endDate')
    ax.set_ylabel('Mercados')
    ax.legend()

    ax = axes[1]
    splits = ['Train', 'Val']
    pos_rates = [100*y_train.mean(), 100*y_val.mean()]
    ax.bar(splits, pos_rates, color=[PALETTE[0], PALETTE[1]], alpha=0.85, width=0.4)
    ax.axhline(100*y.mean(), color='gray', linestyle='--', lw=1.5, label=f'Global {100*y.mean():.1f}%')
    for i, (s, r) in enumerate(zip(splits, pos_rates)):
        ax.text(i, r + 0.3, f'{r:.1f}%', ha='center')
    ax.set_ylabel('% Positivos (Yes)')
    ax.set_title('% Yes en train vs val\n(debe ser similar — no hay fuga temporal)')
    ax.legend()
    ax.set_ylim(0, 50)

    plt.suptitle('Split temporal — train/val', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES / 'processed_temporal_split.png', bbox_inches='tight')
    plt.show()

    print(f'Train: {len(train_idx):,} samples | {100*y_train.mean():.1f}% Yes')
    print(f'Val:   {len(val_idx):,} samples  | {100*y_val.mean():.1f}% Yes')
else:
    print('No hay timestamps disponibles para split temporal')

## 9. Resumen ejecutivo del dataset procesado

In [ ]:
print('=' * 60)
print('RESUMEN DEL DATASET PROCESADO')
print('=' * 60)
print(f'\nSamples totales:        {N:,}')
print(f'Features numéricas:     {F}')
print(f'Positivos (Yes):        {int(y.sum()):,} ({100*y.mean():.1f}%)')
print(f'Negativos (No):         {int((1-y).sum()):,} ({100*(1-y).mean():.1f}%)')

zero_heavy = [(name, pct) for name, pct in zip(NUMERICAL_FEATURE_NAMES, (X==0).mean(axis=0)*100) if pct > 50]
print(f'\nFeatures con >50% ceros: {len(zero_heavy)}')
for name, pct in sorted(zero_heavy, key=lambda x: -x[1]):
    print(f'  {name}: {pct:.1f}%')

print(f'\nTop 5 features por AUC-ROC univariado:')
for name, auc in auc_series.head(5).items():
    print(f'  {name}: {auc:.3f}')

print('\nNotas para el modelo:')
print('  - neg_risk tiene alta separabilidad (confirmar con AUC arriba)')
print('  - Features TS útiles solo donde hay price_history (coverage ~70-80%)')
print('  - Liquidity/order book = 0 en training (solo resueltos). OK para entrenamiento.')
print('  - Split temporal: train=mercados viejos, val=mercados recientes')
print('=' * 60)